## Create map

### Packages

In [2]:
from resplotlib import rpc
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import numpy as np
import pandas as pd

### Functions

In [3]:
# Functions
def buffer_aoi(gdf_aoi, buffer=1000):
    # Reproject to EPSG:3857
    crs = gdf_aoi.crs
    gdf_aoi = gdf_aoi.to_crs("EPSG:3857")

    # Buffer area of interest
    gdf_aoi["geometry"] = gdf_aoi.buffer(buffer)

    # Reproject back to original crs
    gdf_aoi = gdf_aoi.to_crs(crs)

    # Return buffered aoi
    return gdf_aoi


def features_from_aoi(gdf_aoi, tags, clip="false"):
    # Get bounds
    bounds = gdf_aoi.total_bounds

    # Get features from bbox
    try:
        gdf_features = ox.features_from_bbox(bounds, tags)
    except ox._errors.InsufficientResponseError:
        gdf_features = gpd.GeoDataFrame(columns=["geometry"], crs=gdf_aoi.crs)

    # Clip features to bbox
    if clip == "aoi":
        gdf_features = gdf_features.clip(gdf_aoi)
    if clip == "box":
        gdf_features = gdf_features.cx[bounds[0] : bounds[2], bounds[1] : bounds[3]]

    # Return features
    return gdf_features


### Settings

In [4]:
# Define geographic area of interest
geocode_aoi = "Amsterdam, Netherlands"

# Define colour palette
colours = {"aoi": "black", "suburb": "grey", "train": "#EC3339", "metro": "#28A3D9", "light_rail": "#1DA44D", "tram": "#F5793B"}

### Get area of interest

In [5]:
# Get area of interest
gdf_aoi = ox.geocode_to_gdf(geocode_aoi)

# Get buffered area of interest
gdf_buffer = buffer_aoi(gdf_aoi, buffer=500)

# Get bounding box
gdf_bbox = gpd.GeoDataFrame(data={"geometry": [box(*gdf_buffer.total_bounds)], "name": ["bbox"]}, crs=gdf_buffer.crs)

# Get bounds, centre, width and height
bounds = gdf_bbox.total_bounds
centre = (bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2
width = bounds[2] - bounds[0]
height = bounds[3] - bounds[1]

# Get aspect ratio from plot
fig, ax = plt.subplots(figsize=(width / 100, height / 100))
gdf_buffer.boundary.plot(ax=ax)
aspect_ratio = ax.get_aspect()
plt.close(fig)

# Adjust width and height based on aspect ratio to fit onto A4 paper
if width > height:
    if width < height * aspect_ratio * np.sqrt(2):
        width = height * aspect_ratio * np.sqrt(2)
    else:
        height = width / aspect_ratio / np.sqrt(2)
elif height > width:
    if height < width / aspect_ratio * np.sqrt(2):
        height = width / aspect_ratio * np.sqrt(2)
    else:
        width = height * aspect_ratio / np.sqrt(2)
bounds = centre[1] - width / 2, centre[0] - height / 2, centre[1] + width / 2, centre[0] + height / 2
gdf_bbox = gpd.GeoDataFrame(data={"geometry": [box(*bounds)], "name": ["bbox"]}, crs=gdf_bbox.crs)

# Get limits
bounds = gdf_bbox.total_bounds
xlim = (bounds[0], bounds[2])
ylim = (bounds[1], bounds[3])

# Show area of ineterst
rpc.explore_geometries(gdf=gdf_aoi)

Map(center=[np.float64(52.354619), np.float64(4.9039699)], controls=(ZoomControl(options=['position', 'zoom_in…

### Get assets

In [6]:
# Get suburbs
gdf_suburbs = features_from_aoi(gdf_bbox, tags={"place": "suburb"})
gdf_suburbs = gdf_suburbs.clip(gdf_aoi).reset_index(drop=True)
gdf_suburbs = gdf_suburbs[np.logical_or(gdf_suburbs.geom_type == "Polygon", gdf_suburbs.geom_type == "MultiPolygon")]
gdf_suburbs["type"] = "suburb"
gdf_suburbs = gdf_suburbs[["name", "type", "geometry"]]

In [7]:
# Get train lines
gdf_train_lines = features_from_aoi(gdf_bbox, tags={"railway": "rail"})
if "usage" in gdf_train_lines.columns:
    gdf_train_lines = gdf_train_lines[gdf_train_lines["usage"].isin(["main", "branch"])]
gdf_train_lines["type"] = "train"

# Get metro lines
gdf_metro_lines = features_from_aoi(gdf_bbox, tags={"railway": "subway"})
if "service" in gdf_metro_lines.columns:
    gdf_metro_lines = gdf_metro_lines[~gdf_metro_lines["service"].isin(["yard", "siding", "spur", "crossover"])]
gdf_metro_lines["type"] = "metro"

# Get light rail lines
gdf_light_rail_lines = features_from_aoi(gdf_bbox, tags={"railway": "light_rail"})
gdf_light_rail_lines["type"] = "light_rail"

# Get tram lines
gdf_tram_lines = features_from_aoi(gdf_bbox, tags={"railway": "tram"})
if "service" in gdf_tram_lines.columns:
    gdf_tram_lines = gdf_tram_lines[~gdf_tram_lines["service"].isin(["yard", "siding", "spur", "crossover"])]
gdf_tram_lines = gdf_tram_lines[["name", "geometry"]]
gdf_tram_lines["type"] = "tram"

# Merge lines
gdf_lines_ls = [gdf_train_lines, gdf_metro_lines, gdf_light_rail_lines, gdf_tram_lines]
gdf_lines = gpd.GeoDataFrame(pd.concat(gdf_lines_ls, ignore_index=True), crs=gdf_bbox.crs)
gdf_lines["rank"] = gdf_lines["type"].map({"train": 0, "metro": 1, "light_rail": 2, "tram": 3})
gdf_lines = gdf_lines[["name", "type", "rank", "geometry"]]

In [8]:
# Get stations
gdf_stations = features_from_aoi(gdf_bbox, tags={"railway": "station"})
gdf_stations["geometry"] = gdf_stations["geometry"].map(lambda x: x.centroid if x.geom_type != "Point" and x.geom_type != "MultiPoint" else x)

# Get train stations
gdf_train_stations = gdf_stations[~gdf_stations["station"].isin(["subway", "light_rail"])].copy()
gdf_train_stations["type"] = "train"

# Get metro stations
gdf_metro_stations = gdf_stations[gdf_stations["station"].isin(["subway"])].copy()
gdf_metro_stations["type"] = "metro"

# Get light rail stations
gdf_light_rail_stations = gdf_stations[gdf_stations["station"].isin(["light_rail"])].copy()
gdf_light_rail_stations["type"] = "light_rail"


# Get tram stations
gdf_tram_stations = features_from_aoi(gdf_bbox, tags={"railway": "tram_stop"})
gdf_tram_stations["geometry"] = gdf_tram_stations["geometry"].map(lambda x: x.centroid if x.geom_type != "Point" else x)
gdf_tram_stations["type"] = "tram"

# Remove museum tramstation (for Amsterdam)
gdf_tram_stations = gdf_tram_stations[gdf_tram_stations["wikidata"].notna()]

# Merge stations
gdf_stations_ls = [gdf_train_stations, gdf_metro_stations, gdf_light_rail_stations, gdf_tram_stations]
gdf_stations = gpd.GeoDataFrame(pd.concat(gdf_stations_ls, ignore_index=True), crs=gdf_bbox.crs)
gdf_stations["rank"] = gdf_stations["type"].map({"train": 0, "metro": 1, "light_rail": 2, "tram": 3})
gdf_stations = gdf_stations[["name", "type", "rank", "geometry"]]

In [9]:
# Copy stations
gdf_stations2 = gdf_stations.copy()

# Remove aliases
alias_dict = {
    "Amsterdam Centraal": ["Centraal Station", "Centraal"],
    "Amsterdam Zuid": ["Station Zuid", "Zuid"],
    "Amsterdam Amstel": ["Station Amstel", "Amstel"],
    "Amsterdam Sloterdijk": ["Station Sloterdijk", "Sloterdijk"],
    "Amsterdam Lelylaan": ["Station Lelylaan", "Lelylaan"],
    "Amserdam RAI": ["Station RAI", "RAI"],
}
for name, aliases in alias_dict.items():
    for alias in aliases:
        gdf_stations2.loc[gdf_stations2["name"] == alias, "name"] = name

# Remove duplicates, keeping the one with the highest rank (train > metro > light rail > tram)
gdf_stations2 = gdf_stations2.sort_values("rank").drop_duplicates(subset=["name"], keep="first").reset_index(drop=True)
gdf_stations2 = gdf_stations2[["name", "type", "rank", "geometry"]]

In [10]:
def points_to_circles(gdf_points, radius=100):
    gdf_circles = gdf_points.copy()
    gdf_circles = gdf_circles.to_crs(gdf_points.estimate_utm_crs())
    gdf_circles["geometry"] = gdf_circles.buffer(radius)
    gdf_circles = gdf_circles.to_crs(gdf_points.crs)
    return gdf_circles


# Get hiding areas
gdf_hide_areas = points_to_circles(gdf_stations2, radius=300)

In [30]:
from ipyleaflet import Map, basemaps, LayersControl, Popup, GeoJSON, GeomanDrawControl
from ipywidgets import HTML, jslink


popup = Popup(close_button=True, auto_close=False, auto_pan=False, close_on_escape_key=False)

hide_area = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "black", "fillColor": "none", "opacity": 1, "weight": 2},
    hover_style={"color": "black", "fillColor": "none", "opacity": 1, "weight": 2},
)


def on_click_handler(**kwargs):
    properties = kwargs["properties"]
    popup.child = HTML(properties["name"])
    popup.location = kwargs["coordinates"]  # This is what this PR provides.
    popup.open_popup()
    # gdf_hide_area = gpd.GeoDataFrame.from_features([feature], crs=gdf_stations2.crs)
    # gdf_hide_area = gdf_hide_area.to_crs("EPSG:3857")
    # gdf_hide_area["geometry"] = gdf_hide_area.buffer(500)
    # gdf_hide_area = gdf_hide_area.to_crs("EPSG:4326")
    # hide_area.data = {"type": "FeatureCollection", "features": list(gdf_hide_area.iterfeatures())}


m = Map(center=centre, zoom=11, basemap=basemaps.CartoDB.Positron, scroll_wheel_zoom=True, layout={"width": "100%", "height": "600px"})
m.add_control(LayersControl(position="topright"))

# Area of interest
m = rpc.explore_geometries(
    gdf=gdf_aoi,
    m=m,
    style_kwargs={"color": "black", "fillColor": "none"},
    hover_style_kwargs={"color": "black", "fillColor": "none"},
    name="Area of interest",
)

# Suburbs
m = rpc.explore_geometries(
    gdf=gdf_aoi,
    m=m,
    style_kwargs={"color": "black", "fillColor": "none"},
    hover_style_kwargs={"color": "black", "fillColor": "none"},
    name="Suburbs",
)

# Lines
gdf_groups = gdf_lines.groupby("type")
for name, group in gdf_groups:
    m = rpc.explore_geometries(
        gdf=group,
        m=m,
        style_kwargs={"color": colours[name], "fillColor": "none", "opacity": 1, "weight": 1},
        hover_style_kwargs={"color": colours[name], "fillColor": "none", "opacity": 1, "weight": 1},
        name=f"{name.capitalize()} lines",
        on_click=False,
    )

# Stations
gdf_groups = gdf_stations2.groupby("type")


for name, group in gdf_groups:
    m = rpc.explore_geometries(
        gdf=group,
        m=m,
        style_kwargs={"color": colours[name], "fillColor": colours[name], "opacity": 1, "weight": 1},
        hover_style_kwargs={"color": colours[name], "fillColor": colours[name], "opacity": 1, "weight": 1},
        name=f"{name.capitalize()} stations",
    )
    layer = m.layers[-1]
    layer.on_click(on_click_handler)


rpc.explore_geometries(
    gdf_hide_areas,
    m=m,
    style_kwargs={"color": "black", "fillColor": "none", "opacity": 1, "weight": 2},
    hover_style_kwargs={"color": "black", "fillColor": "none", "opacity": 1, "weight": 2},
    name="Hiding areas",
    style_function=lambda feature: {"color": "black", "fillColor": "none", "opacity": 1, "weight": 2},
)
m.add(popup)

m

Map(center=[np.float64(52.354614030228845), np.float64(4.903969624042253)], controls=(ZoomControl(options=['po…